# AI Traffic Violation Detection — YOLO26n vs YOLOv8n Training on Google Colab (T4 GPU)

Trains **both** models on the same dataset with the same settings, so their accuracy can be compared (Review 1 planned YOLOv8; the project uses YOLO26).

### Step-by-Step Instructions:
1. **Select GPU Hardware Accelerator**:
   - Click **Runtime** in the top navigation bar.
   - Click **Change runtime type**.
   - Under *Hardware accelerator*, select **T4 GPU**.
   - Click **Save**.
2. **Upload Dataset**:
   - Create `datasets/merged.zip` on your local PC using `python training/zip_dataset.py`.
   - Upload `merged.zip` to your Google Drive root directory (`MyDrive/merged.zip`).
3. **Run All Cells Below** in order.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("Google Drive mounted successfully!")

## 2. Install Latest Ultralytics with YOLO26 Support

In [ ]:
!pip install -U ultralytics
import ultralytics
ultralytics.checks()

## 3. Locate & Unzip Dataset from Google Drive

In [ ]:
import os
import zipfile
import shutil
from pathlib import Path

# Check standard locations in Google Drive for merged.zip
drive_candidates = [
    Path('/content/drive/MyDrive/merged.zip'),
    Path('/content/drive/MyDrive/traffic_dataset/merged.zip'),
    Path('/content/drive/MyDrive/datasets/merged.zip'),
]

zip_source = None
for cand in drive_candidates:
    if cand.exists():
        zip_source = cand
        break

if zip_source is None:
    # Recursive search if placed in a subfolder
    print("Searching Drive for merged.zip...")
    matches = list(Path('/content/drive/MyDrive').rglob('merged.zip'))
    if matches:
        zip_source = matches[0]

if zip_source is None or not zip_source.exists():
    raise FileNotFoundError("merged.zip not found in Google Drive! Please upload it to your Google Drive root (MyDrive/merged.zip).")

print(f"Found dataset at: {zip_source}")
extract_dest = Path('/content/datasets/merged')
extract_dest.mkdir(parents=True, exist_ok=True)

print(f"Extracting dataset to {extract_dest}...")
with zipfile.ZipFile(zip_source, 'r') as z:
    z.extractall(extract_dest)

# Verify data.yaml
yaml_path = extract_dest / 'data.yaml'
yaml_content = f"""# YOLO26 Traffic Detection Dataset
path: {extract_dest}
train: images/train
val: images/val

nc: 3
names:
  0: helmet
  1: no_helmet
  2: number_plate
"""
yaml_path.write_text(yaml_content.strip() + '\n')
print("Configured data.yaml:")
print(yaml_path.read_text())

train_imgs = len(list((extract_dest / 'images' / 'train').glob('*.*')))
val_imgs = len(list((extract_dest / 'images' / 'val').glob('*.*')))
print(f"Dataset ready: {train_imgs} train images, {val_imgs} val images.")

## 4. Train YOLO26n and YOLOv8n on the same dataset
- Same data, epochs (40), batch (16), image size (640), patience (10) and seed for both, so the comparison is fair
- Expected duration on T4 GPU: ~35–55 minutes **per model**
- Short on Colab time? Remove one name from `MODELS_TO_TRAIN` and run it in a second session — results are saved to Drive per model


In [ ]:
from ultralytics import YOLO

MODELS_TO_TRAIN = ['yolo26n.pt', 'yolov8n.pt']   # the project model first, then the Review 1 baseline
# Add any model from the dashboard's Models page to compare more, e.g. 'yolo11n.pt', 'yolov10n.pt', 'yolov5nu.pt',
# 'yolo26s.pt' (each nano model adds ~35-55 min on a T4; small models take longer)
SETTINGS = dict(data='/content/datasets/merged/data.yaml', epochs=40, imgsz=640, batch=16, patience=10,
                device=0, seed=0, project='/content/runs/train', save=True, plots=True, verbose=True)

trained = {}
for base in MODELS_TO_TRAIN:
    name = base.replace('.pt', '') + '_traffic'
    print(f"\n========== Training {base} -> runs/train/{name} ==========")
    model = YOLO(base)
    model.train(name=name, exist_ok=True, **SETTINGS)
    trained[base] = name
print("Training complete:", trained)


## 5. Evaluate both & export results to Google Drive
For each model: `best.pt`, `results.csv`, charts and `evaluation_results_<model>.json` (precision, recall, mAP per class) go to `MyDrive/traffic_model_output/<model>/`. A side-by-side table is printed at the end.

On your PC: copy `yolo26n/best.pt` → `models/traffic_custom.pt`, and both `evaluation_results_*.json` files → `data/`. The dashboard's **Evaluation** page then shows YOLOv8n and YOLO26n side by side.


In [ ]:
import json
import shutil
from pathlib import Path
from ultralytics import YOLO

drive_root = Path('/content/drive/MyDrive/traffic_model_output')
drive_root.mkdir(parents=True, exist_ok=True)
class_names = ['helmet', 'no_helmet', 'number_plate']
summary = {}

for base, name in trained.items():
    stem = base.replace('.pt', '')
    run_dir = Path('/content/runs/train') / name
    out_dir = drive_root / stem
    out_dir.mkdir(parents=True, exist_ok=True)
    best_weights = run_dir / 'weights' / 'best.pt'
    if not best_weights.exists():
        print(f"Warning: {best_weights} not found, skipping {base}")
        continue
    shutil.copy(best_weights, out_dir / 'best.pt')
    for fname in ['results.csv', 'confusion_matrix.png', 'confusion_matrix_normalized.png', 'results.png', 'F1_curve.png', 'PR_curve.png']:
        if (run_dir / fname).exists():
            shutil.copy(run_dir / fname, out_dir / fname)

    print(f"\nValidating {base} ...")
    val = YOLO(str(best_weights)).val(data='/content/datasets/merged/data.yaml', device=0)
    eval_data = {
        "model": base,
        "overall": {
            "precision": round(float(val.results_dict.get('metrics/precision(B)', 0)), 4),
            "recall": round(float(val.results_dict.get('metrics/recall(B)', 0)), 4),
            "mAP50": round(float(val.results_dict.get('metrics/mAP50(B)', 0)), 4),
            "mAP50_95": round(float(val.results_dict.get('metrics/mAP50-95(B)', 0)), 4),
        },
        "speed_ms_per_image": {k: round(float(v), 2) for k, v in getattr(val, 'speed', {}).items()},
        "per_class": {},
    }
    for i, cls_name in enumerate(class_names):
        try:
            eval_data["per_class"][cls_name] = {
                "precision": round(float(val.box.p[i]), 4),
                "recall": round(float(val.box.r[i]), 4),
                "mAP50": round(float(val.box.ap50[i]), 4),
                "mAP50_95": round(float(val.box.ap[i]), 4),
            }
        except Exception as e:
            print(f"Could not extract class {cls_name}: {e}")
    for target in [out_dir / f'evaluation_results_{stem}.json', drive_root / f'evaluation_results_{stem}.json']:
        target.write_text(json.dumps(eval_data, indent=2))
    if stem == 'yolo26n':  # the project model is also the default results file
        (drive_root / 'evaluation_results.json').write_text(json.dumps(eval_data, indent=2))
        shutil.copy(best_weights, drive_root / 'traffic_custom.pt')
    summary[stem] = eval_data["overall"]

print("\n================ YOLO26n vs YOLOv8n (same dataset) ================")
print(f"{'metric':12}" + "".join(f"{m:>12}" for m in summary))
for k in ["precision", "recall", "mAP50", "mAP50_95"]:
    print(f"{k:12}" + "".join(f"{summary[m][k]:>12.4f}" for m in summary))
print("\nAll outputs saved to", drive_root)


## 6. (Optional) Train YOLO26 Small (`yolo26s.pt`)
Higher capacity and accuracy, ~2x slower training. Uncomment and run if Colab quota and time permits.

In [ ]:
# Optional: Train yolo26s.pt
# model_s = YOLO('yolo26s.pt')
# results_s = model_s.train(
#     data='/content/datasets/merged/data.yaml',
#     epochs=40,
#     imgsz=640,
#     batch=16,
#     patience=10,
#     device=0,
#     project='/content/runs/train',
#     name='yolo26s_traffic',
#     save=True,
#     plots=True,
# )
# print("YOLO26s Training complete!")